# Stock Market Analysis

An original, research-oriented notebook for exploring historical prices, return behavior, drawdowns, and Monte Carlo price paths. Market history is descriptive and scenario-based, not investment advice.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path('data/stock_history.csv')
if not data_path.exists():
    raise FileNotFoundError('Add the real Kaggle stock history CSV to data/stock_history.csv')
prices = pd.read_csv(data_path, parse_dates=['Date']).rename(columns={'Ticker': 'Symbol'})
prices.head()

## Data quality and shape

The first pass checks dates, duplicate observations, missing values, and the available symbols. Keeping these checks visible makes market conclusions easier to reproduce.

In [ ]:
prices.info()
display(prices.isna().sum())
display(prices['Symbol'].value_counts())

In [ ]:
prices = prices.sort_values(['Symbol', 'Date']).drop_duplicates(['Symbol', 'Date'])
prices['daily_return'] = prices.groupby('Symbol')['Close'].pct_change()
prices['rolling_volatility'] = prices.groupby('Symbol')['daily_return'].transform(lambda values: values.rolling(30).std() * np.sqrt(252))
prices['drawdown'] = prices['Close'] / prices.groupby('Symbol')['Close'].cummax() - 1
prices.describe(include='all')

## Price and risk view

In [ ]:
for symbol, group in prices.groupby('Symbol'):
    figure, axis = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axis[0].plot(group['Date'], group['Close'], label=symbol, color='#315f78')
    axis[0].set_title(f'{symbol}: closing price')
    axis[1].plot(group['Date'], group['drawdown'], color='#bf4f3f')
    axis[1].set_title('Drawdown from running high')
    axis[1].set_ylabel('Drawdown')
    plt.tight_layout()
    plt.show()

In [ ]:
risk_summary = prices.groupby('Symbol').agg(
    annualized_return=('daily_return', lambda values: values.mean() * 252),
    annualized_volatility=('daily_return', lambda values: values.std() * np.sqrt(252)),
    worst_drawdown=('drawdown', 'min'),
)
risk_summary

## Monte Carlo scenario

This simulation resamples the return distribution to show a range of possible paths. It is not a forecast and does not model changing regimes, liquidity, fees, or news shocks.

In [ ]:
symbol = prices['Symbol'].iloc[0]
history = prices.loc[prices['Symbol'].eq(symbol), 'daily_return'].dropna()
rng = np.random.default_rng(42)
steps, simulations = 252, 500
sampled_returns = rng.choice(history.to_numpy(), size=(steps, simulations), replace=True)
paths = prices.loc[prices['Symbol'].eq(symbol), 'Close'].iloc[-1] * np.exp(np.cumsum(np.log1p(sampled_returns), axis=0))
percentiles = np.percentile(paths, [5, 50, 95], axis=1)
plt.figure(figsize=(12, 5))
plt.plot(percentiles.T, color=['#bf4f3f', '#315f78', '#d9984a'])
plt.title(f'{symbol}: 12-month resampled return scenarios')
plt.ylabel('Simulated price')
plt.show()
pd.Series({'p05': percentiles[0, -1], 'median': percentiles[1, -1], 'p95': percentiles[2, -1]})

## Decision notes

A responsible readout should compare return with volatility and drawdown, show the date range, disclose survivorship and look-ahead risks, and avoid turning a scenario range into a promise. A production version would add corporate actions, benchmark comparison, transaction costs, and walk-forward validation.